In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1-P_IjZjxISRXFW4DLbMF3atDbu5CXgGK", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_00_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Imports Plots
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_imports_plots.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Notebook Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_01_notebook_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# 🔍 Retrieval Strategies: Dense, Sparse, and Hybrid

*Part 4 of 5 in the Vizuara series on RAG Systems (Context Engineering Course)*

**Estimated time: 45 minutes**

In the previous notebook, we built a complete RAG pipeline with dense (embedding-based) retrieval. But dense retrieval has a blind spot: it misses exact keyword matches. Meanwhile, classic keyword search (BM25) has the opposite blind spot — it misses semantic meaning. The best production systems combine both.

By the end of this notebook, you will have built all four retrieval strategies from scratch — dense, sparse, hybrid, and reranked — and measured exactly when each one wins.

In [ ]:
#@title 🎧 Listen: Ai Ta
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_ai_ta.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/rag-systems/practice/4/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Section0 Transition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_section0_transition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 0. Setup and Installations

Let us install the libraries we need. We use `sentence-transformers` for dense embeddings and cross-encoder reranking, `rank-bm25` for sparse retrieval, and `faiss-cpu` for fast vector search.

In [ ]:
#@title 🎧 Listen: Install Libs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_install_libs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
!pip install -q sentence-transformers rank-bm25 faiss-cpu numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import textwrap
import time
import re
from typing import List, Tuple, Dict, Optional
from collections import Counter

# Reproducibility
np.random.seed(42)

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✅ Setup complete!")

In [ ]:
#@title 🎧 Listen: Why Matter Analogy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_why_matter_analogy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. 🤔 Why Does This Matter?

Think of a librarian. Some librarians are incredible at understanding *what you mean* — you say "I need something about automobile regulations" and they find a book titled "Federal Motor Vehicle Safety Standards" even though you never said "motor vehicle." That is **dense retrieval**: it understands semantics.

Other librarians are incredibly precise with call numbers and indices — you say "Find error code ERR-4021" and they locate the exact page in seconds. But ask them about "common server problems" and they are lost because that phrase does not appear anywhere verbatim. That is **sparse retrieval (BM25)**: it matches exact terms.

A librarian who can do *both* — understand your topic AND find specific call numbers — is strictly better than one who only does one. That is **hybrid retrieval**.

But we can go further. After gathering candidates from both approaches, we can hire a **specialist** to carefully read each candidate and re-score them. That is **cross-encoder reranking** — a second, more expensive pass that dramatically improves precision.

**The complete production pipeline:**
1. **Dense retrieval** (semantic understanding) → top-50 candidates
2. **Sparse retrieval** (keyword matching) → top-50 candidates
3. **Hybrid fusion** (combine both lists) → merged top-50
4. **Cross-encoder reranking** (careful re-scoring) → final top-5

Let us build every piece.

In [ ]:
#@title 🎧 Listen: Intuition Scenarios
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_intuition_scenarios.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. 🧠 Building Intuition

### Three Scenarios That Reveal the Gaps

**NO CODE in this section** — just thinking.

---

**Scenario 1: Semantic query, no keyword overlap**

> Query: "automobile regulations in the United States"

Your corpus contains a document: *"The National Highway Traffic Safety Administration (NHTSA) enforces federal motor vehicle safety standards under 49 CFR Part 571."*

- **Dense retrieval** finds it — "automobile regulations" and "motor vehicle safety standards" land in similar embedding space.
- **BM25 misses it** — not a single query word ("automobile," "regulations," "United," "States") appears in the document.
- **Winner: Dense**

---

**Scenario 2: Exact keyword query, no semantic match**

> Query: "error code ERR-4021 connection timeout"

Your corpus contains: *"ERR-4021: TCP socket connection timed out after 30 seconds. Retry with exponential backoff."*

- **BM25 finds it instantly** — "ERR-4021" is an exact token match with extremely high IDF (rare term).
- **Dense retrieval retrieves garbage** — it finds documents about "error handling best practices" and "timeout configuration," which are semantically related but not the right document.
- **Winner: Sparse (BM25)**

---

**Scenario 3: Mixed query, needs both**

> Query: "how to fix ERR-4021 in a microservices architecture"

This query has both a precise identifier ("ERR-4021") and a conceptual component ("microservices architecture"). Neither dense nor sparse alone gets the full picture. Hybrid search combines the candidate lists and finds documents that match on keywords AND semantics.

- **Winner: Hybrid**

---

### Cross-Encoder Reranking: The Second Opinion

After hybrid retrieval gives us ~50 candidates, we still have a ranking problem. The fusion scores are heuristic — they combine two different scoring scales. A **cross-encoder** solves this by taking each (query, document) pair and scoring them *jointly*. Unlike bi-encoders that embed query and document independently, a cross-encoder feeds the concatenated pair through a transformer, allowing full token-level interaction.

Think of it as: bi-encoder retrieval is speed-dating (quick impression), cross-encoder reranking is a follow-up interview (careful evaluation).

The tradeoff is cost: a cross-encoder must process each (query, doc) pair separately, so running it on 10,000 documents is expensive. That is why we use the two-stage pattern: **bi-encoder top-50 → cross-encoder top-5**.

In [ ]:
#@title 🎧 Listen: Bm25 Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_bm25_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. 📐 The Mathematics

### 3.1 BM25 Scoring Formula

BM25 (Best Matching 25) is the gold standard of keyword retrieval. For a query $Q$ containing terms $q_1, q_2, \ldots, q_n$ and a document $D$:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

**What does each piece mean computationally?**

- $f(q_i, D)$ = how many times query term $q_i$ appears in document $D$ (term frequency)
- $|D|$ = document length (in words)
- $\text{avgdl}$ = average document length across the corpus
- $k_1$ = controls term frequency saturation (typically 1.2-2.0). Higher $k_1$ means more credit for repeated terms.
- $b$ = controls length normalization (typically 0.75). When $b = 1$, long documents are heavily penalized. When $b = 0$, length is ignored.

The IDF (Inverse Document Frequency) component:

$$\text{IDF}(q_i) = \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$$

where $N$ = total documents, $n(q_i)$ = documents containing term $q_i$.

**Key intuition:** Rare terms get high IDF (they are discriminative). Common terms get low IDF (they are noise). A term appearing in 1 out of 1000 documents is far more useful for retrieval than a term appearing in 900 out of 1000.

In [ ]:
#@title 🎧 Listen: Bm25 Example
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_bm25_example.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Worked Example: BM25 by Hand

Let us compute BM25 for a tiny corpus so every number is traceable.

**Corpus** (3 documents):
- $D_1$: "the cat sat on the mat" (6 words)
- $D_2$: "the dog chased the cat around" (6 words)
- $D_3$: "a fish swam in the pond" (6 words)

**Query**: "cat mat"

**Parameters**: $k_1 = 1.5$, $b = 0.75$, $N = 3$, $\text{avgdl} = 6$

For term "cat": $n(\text{cat}) = 2$ (appears in $D_1$ and $D_2$)

$$\text{IDF}(\text{cat}) = \ln\left(\frac{3 - 2 + 0.5}{2 + 0.5} + 1\right) = \ln\left(\frac{1.5}{2.5} + 1\right) = \ln(1.6) \approx 0.47$$

For term "mat": $n(\text{mat}) = 1$ (appears only in $D_1$)

$$\text{IDF}(\text{mat}) = \ln\left(\frac{3 - 1 + 0.5}{1 + 0.5} + 1\right) = \ln\left(\frac{2.5}{1.5} + 1\right) = \ln(2.667) \approx 0.98$$

Notice: "mat" has higher IDF than "cat" because it is rarer — exactly what we want.

For $D_1$ (contains both "cat" once and "mat" once, $|D_1| = 6$):

$$\text{BM25}(D_1) = 0.47 \cdot \frac{1 \cdot 2.5}{1 + 1.5 \cdot (1 - 0.75 + 0.75 \cdot 1)} + 0.98 \cdot \frac{1 \cdot 2.5}{1 + 1.5 \cdot 1} = 0.47 \cdot 1.0 + 0.98 \cdot 1.0 = 1.45$$

In [ ]:
#@title 🎧 Listen: Bm25 Example Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_bm25_example_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import math

# Verify our hand computation
N = 3
avgdl = 6.0
k1 = 1.5
b = 0.75

# IDF values
def bm25_idf(N, n_qi):
    return math.log((N - n_qi + 0.5) / (n_qi + 0.5) + 1)

idf_cat = bm25_idf(3, 2)
idf_mat = bm25_idf(3, 1)

print(f"IDF('cat') = {idf_cat:.4f}  (appears in 2/3 docs — less discriminative)")
print(f"IDF('mat') = {idf_mat:.4f}  (appears in 1/3 docs — more discriminative)")

# BM25 term score for D1
def bm25_term_score(tf, idf, doc_len, avgdl, k1, b):
    numerator = tf * (k1 + 1)
    denominator = tf + k1 * (1 - b + b * doc_len / avgdl)
    return idf * numerator / denominator

score_cat_d1 = bm25_term_score(tf=1, idf=idf_cat, doc_len=6, avgdl=6, k1=1.5, b=0.75)
score_mat_d1 = bm25_term_score(tf=1, idf=idf_mat, doc_len=6, avgdl=6, k1=1.5, b=0.75)

print(f"\nD1 score for 'cat': {score_cat_d1:.4f}")
print(f"D1 score for 'mat': {score_mat_d1:.4f}")
print(f"D1 total BM25:      {score_cat_d1 + score_mat_d1:.4f}")

# D2 has "cat" but not "mat"
score_cat_d2 = bm25_term_score(tf=1, idf=idf_cat, doc_len=6, avgdl=6, k1=1.5, b=0.75)
print(f"\nD2 total BM25:      {score_cat_d2:.4f}  (only matches 'cat')")

# D3 has neither
print(f"D3 total BM25:      0.0000  (no query terms)")

print(f"\n💡 D1 wins because it matches BOTH query terms. The rare term 'mat' contributes more.")

In [ ]:
#@title 🎧 Listen: Rrf Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_rrf_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 3.2 Reciprocal Rank Fusion (RRF)

When we have two ranked lists (one from dense, one from sparse), how do we merge them? **Reciprocal Rank Fusion** is elegant:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

where $R$ is the set of ranking systems (dense and sparse), $\text{rank}_r(d)$ is document $d$'s rank in system $r$, and $k$ is a smoothing constant (typically 60).

**Worked example:**

| Document | Dense Rank | Sparse Rank | RRF Score (k=60) |
|----------|-----------|-------------|-------------------|
| Doc A | 1 | 5 | $\frac{1}{61} + \frac{1}{65} = 0.01639 + 0.01538 = 0.03177$ |
| Doc B | 3 | 1 | $\frac{1}{63} + \frac{1}{61} = 0.01587 + 0.01639 = 0.03227$ |
| Doc C | 2 | 10 | $\frac{1}{62} + \frac{1}{70} = 0.01613 + 0.01429 = 0.03042$ |

**Result:** Doc B wins! It was rank 3 in dense and rank 1 in sparse. RRF rewards documents that appear high in *multiple* lists, not just one.

**Why $k = 60$?** The parameter $k$ controls how much we care about exact rank position. Large $k$ makes the formula more "democratic" — the difference between rank 1 and rank 5 is small. Small $k$ makes it more "winner-take-all" — rank 1 gets much more weight.

In [ ]:
#@title 🎧 Listen: Encoder Arch
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_encoder_arch.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 3.3 Cross-Encoder vs Bi-Encoder Architecture

Let us clarify the architectural difference computationally:

**Bi-encoder (what we use for dense retrieval):**
- Encode query: $\mathbf{q} = \text{Encoder}(\text{query})$ → 384-dim vector
- Encode each document: $\mathbf{d}_i = \text{Encoder}(\text{doc}_i)$ → 384-dim vector
- Score: $\text{sim}(q, d_i) = \cos(\mathbf{q}, \mathbf{d}_i)$
- **Cost**: Encode $N$ docs once (offline), then 1 query encoding + $N$ dot products at query time
- **Speed**: Milliseconds for 1M documents (using FAISS)

**Cross-encoder (what we use for reranking):**
- Concatenate: $\text{input}_i = [\text{CLS}] \; \text{query} \; [\text{SEP}] \; \text{doc}_i \; [\text{SEP}]$
- Score: $s_i = \text{CrossEncoder}(\text{input}_i)$ → single scalar
- **Cost**: One full transformer forward pass per (query, doc) pair
- **Speed**: ~100ms per pair, so 50 pairs = ~5 seconds

The two-stage pattern balances these: bi-encoder narrows 100K documents to 50 candidates in milliseconds, then cross-encoder carefully scores those 50 in a few seconds.

In [ ]:
#@title 🎧 Listen: Rrf Example Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_rrf_example_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verify the RRF worked example from above
def rrf_score_manual(dense_rank: int, sparse_rank: int, k: int = 60) -> float:
    """Compute RRF score for a document given its ranks in two systems."""
    return 1.0 / (k + dense_rank) + 1.0 / (k + sparse_rank)

# Verify the table above
docs_example = {
    'Doc A': (1, 5),
    'Doc B': (3, 1),
    'Doc C': (2, 10),
}

print("RRF Worked Example (k=60):")
print("-" * 55)
print(f"{'Document':<10} {'Dense':>6} {'Sparse':>7} {'RRF Score':>12} {'Rank':>6}")
print("-" * 55)

rrf_scores = {}
for name, (dr, sr) in docs_example.items():
    score = rrf_score_manual(dr, sr, k=60)
    rrf_scores[name] = score

# Sort by RRF score
for rank, (name, score) in enumerate(sorted(rrf_scores.items(), key=lambda x: -x[1]), 1):
    dr, sr = docs_example[name]
    print(f"{name:<10} {dr:>6} {sr:>7} {score:>12.5f} {rank:>6}")

print(f"\n💡 Doc B wins despite being rank 3 in dense — its rank 1 in sparse compensates.")

In [ ]:
#@title 🎧 Listen: Section4 Transition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_section4_transition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. 🔨 Let Us Build It — Component by Component

### 4.1 Creating the Test Corpus

We need a corpus large enough to test retrieval realistically. Let us create 120+ documents spanning technology, medicine, finance, and law — with both short factual entries and longer explanatory passages.

In [ ]:
#@title 🎧 Listen: Corpus Creation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_15_corpus_creation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ========================================
# CORPUS: 120+ documents across 4 domains
# ========================================

tech_docs = [
    "Kubernetes orchestrates containerized applications across clusters of machines, handling deployment, scaling, and failover automatically.",
    "Docker containers package applications with their dependencies into standardized units, ensuring consistent behavior across development and production environments.",
    "GraphQL provides a query language for APIs that lets clients request exactly the data they need, reducing over-fetching compared to REST endpoints.",
    "REST APIs use HTTP methods (GET, POST, PUT, DELETE) to perform CRUD operations on resources identified by URLs.",
    "WebSocket protocol enables full-duplex communication channels over a single TCP connection, essential for real-time applications like chat and live dashboards.",
    "gRPC uses Protocol Buffers for serialization and HTTP/2 for transport, offering high-performance RPC communication between microservices.",
    "Redis is an in-memory key-value store used for caching, session management, and pub/sub messaging with sub-millisecond latency.",
    "PostgreSQL is an open-source relational database supporting advanced features like JSONB columns, full-text search, and window functions.",
    "MongoDB stores data in flexible JSON-like documents, making it popular for applications with evolving schemas and nested data structures.",
    "Apache Kafka is a distributed event streaming platform capable of handling millions of events per second for real-time data pipelines.",
    "Terraform uses declarative configuration files to provision and manage infrastructure across multiple cloud providers.",
    "CI/CD pipelines automate the process of building, testing, and deploying code changes, reducing manual errors and accelerating release cycles.",
    "Microservices architecture decomposes applications into small, independently deployable services that communicate over network protocols.",
    "Service mesh technologies like Istio manage service-to-service communication, providing traffic management, security, and observability.",
    "Error code ERR-4021 indicates a TCP socket connection timeout after 30 seconds. Common fix: increase timeout or implement exponential backoff.",
    "Error code ERR-5033 signals a TLS handshake failure due to certificate mismatch. Verify certificate chain and ensure proper CA configuration.",
    "Error code ERR-2187 means the connection pool is exhausted. Scale up the pool size or investigate connection leaks in the application.",
    "OAuth 2.0 authorization framework delegates user authentication to the service hosting the user account, using access tokens for API authorization.",
    "JWT (JSON Web Tokens) encode claims as a JSON payload signed with a secret or public/private key pair for stateless authentication.",
    "Rate limiting controls the number of API requests a client can make within a time window, preventing abuse and ensuring fair resource usage.",
    "Load balancing distributes incoming network traffic across multiple servers, improving availability and responsiveness of applications.",
    "Content delivery networks (CDNs) cache static assets at edge locations globally, reducing latency for users far from the origin server.",
    "Horizontal scaling adds more machines to handle increased load, while vertical scaling upgrades the existing machine's CPU and memory.",
    "Blue-green deployment maintains two identical production environments, routing traffic to the active one while the other is updated and tested.",
    "Feature flags allow teams to enable or disable features at runtime without deploying new code, enabling gradual rollouts and A/B testing.",
    "Prometheus collects time-series metrics from applications via a pull model, with PromQL for querying and Grafana for visualization.",
    "The CAP theorem states that a distributed system cannot simultaneously guarantee consistency, availability, and partition tolerance.",
    "ACID transactions ensure atomicity, consistency, isolation, and durability in database operations, critical for financial and healthcare systems.",
    "Event sourcing stores all changes as a sequence of immutable events, allowing complete audit trails and temporal queries.",
    "CQRS (Command Query Responsibility Segregation) separates read and write models, optimizing each for its specific workload.",
]

medical_docs = [
    "Metformin is the first-line medication for type 2 diabetes, working by decreasing hepatic glucose production and increasing insulin sensitivity.",
    "CRISPR-Cas9 enables precise genome editing by using guide RNA to direct the Cas9 enzyme to specific DNA sequences for cutting.",
    "mRNA vaccines instruct cells to produce a protein that triggers an immune response, providing immunity without using live virus particles.",
    "PD-1 checkpoint inhibitors block the PD-1 protein on T cells, preventing cancer cells from evading immune detection and destruction.",
    "CAR-T cell therapy engineers a patient's own T cells to express chimeric antigen receptors that target specific cancer cell markers.",
    "Pharmacokinetics describes how the body absorbs, distributes, metabolizes, and excretes drugs, determining appropriate dosing regimens.",
    "The blood-brain barrier selectively restricts the passage of substances from blood to brain tissue, complicating drug delivery for neurological disorders.",
    "Antibiotic resistance develops when bacteria evolve mechanisms to survive exposure to antibiotics, driven by overuse and incomplete treatment courses.",
    "Randomized controlled trials (RCTs) are the gold standard for evaluating treatment efficacy, using random assignment to minimize confounding variables.",
    "The QALY (Quality-Adjusted Life Year) metric combines survival duration with health-related quality of life for health economic evaluations.",
    "Biomarker-driven precision medicine tailors treatment based on individual genetic profiles, particularly in oncology where tumor mutations guide therapy selection.",
    "Clinical trial phase III evaluates drug efficacy and safety in large patient populations, typically requiring 1,000-3,000 participants over 1-4 years.",
    "The FDA 510(k) pathway allows medical devices to reach market by demonstrating substantial equivalence to an existing legally marketed device.",
    "Electronic health records (EHRs) digitize patient medical histories, enabling better care coordination, reduced medical errors, and data-driven clinical decisions.",
    "Telemedicine platforms provide remote clinical services via video conferencing, expanding access to healthcare in rural and underserved areas.",
    "Hemoglobin A1C measures average blood glucose levels over 2-3 months, serving as a key monitoring tool for diabetes management.",
    "Deep learning models for medical imaging achieve radiologist-level accuracy in detecting conditions like diabetic retinopathy and lung nodules.",
    "Drug interaction databases like DrugBank catalog known interactions between medications, helping clinicians avoid harmful combinations.",
    "The human microbiome — trillions of bacteria, viruses, and fungi in the gut — plays a critical role in immune function, metabolism, and mental health.",
    "Sepsis is a life-threatening organ dysfunction caused by the body's dysregulated response to infection, requiring immediate antibiotic therapy and fluid resuscitation.",
]

finance_docs = [
    "The Black-Scholes model prices European options using five inputs: stock price, strike price, time to expiration, risk-free rate, and volatility.",
    "Value at Risk (VaR) estimates the maximum potential loss of a portfolio over a specified time period at a given confidence level.",
    "The Sharpe ratio measures risk-adjusted return by dividing excess return over the risk-free rate by the portfolio's standard deviation.",
    "High-frequency trading (HFT) uses algorithms to execute thousands of trades per second, exploiting tiny price discrepancies across markets.",
    "Compound interest calculates interest on both the initial principal and accumulated interest, described by A = P(1 + r/n)^(nt).",
    "Diversification reduces portfolio risk by spreading investments across different asset classes, sectors, and geographic regions.",
    "The yield curve plots interest rates of bonds with equal credit quality but different maturity dates, with inversions historically predicting recessions.",
    "KYC (Know Your Customer) regulations require financial institutions to verify client identity and assess money laundering risk before establishing relationships.",
    "Basel III framework sets minimum capital requirements for banks, including a 4.5% Common Equity Tier 1 ratio to absorb losses.",
    "Blockchain-based settlement systems promise T+0 trade settlement by eliminating intermediaries, compared to the traditional T+2 cycle.",
    "The Federal Reserve adjusts the federal funds rate to influence borrowing costs, inflation, and employment levels across the economy.",
    "Credit default swaps (CDS) transfer credit risk between parties, functioning as insurance against bond default with premium payments.",
    "Monte Carlo simulation models portfolio risk by generating thousands of random market scenarios based on historical return distributions.",
    "ESG (Environmental, Social, Governance) investing integrates sustainability factors into investment analysis alongside traditional financial metrics.",
    "The efficient market hypothesis states that asset prices fully reflect all available information, making consistent outperformance through stock picking impossible.",
    "Quantitative easing (QE) is an unconventional monetary policy where central banks purchase government bonds to inject liquidity into the financial system.",
    "SWIFT (Society for Worldwide Interbank Financial Telecommunication) processes over 40 million financial messages daily between 11,000+ institutions globally.",
    "AML (Anti-Money Laundering) compliance programs include transaction monitoring, suspicious activity reporting, and customer due diligence procedures.",
    "Robo-advisors use algorithms to build and manage diversified investment portfolios based on client risk tolerance and financial goals.",
    "The Dodd-Frank Act of 2010 reformed financial regulation in response to the 2008 crisis, establishing the CFPB and restricting proprietary trading.",
]

legal_docs = [
    "The General Data Protection Regulation (GDPR) grants EU citizens rights over their personal data, including access, rectification, erasure, and portability.",
    "Section 230 of the Communications Decency Act shields internet platforms from liability for user-generated content while allowing good-faith content moderation.",
    "The EU AI Act classifies AI systems by risk level: unacceptable, high-risk, limited-risk, and minimal-risk, with requirements scaling accordingly.",
    "Patent trolls (non-practicing entities) acquire patents solely to extract licensing fees through litigation threats, without producing goods or services.",
    "Fair use doctrine permits limited use of copyrighted material without permission for purposes like criticism, commentary, education, and parody.",
    "HIPAA (Health Insurance Portability and Accountability Act) establishes standards for protecting sensitive patient health information from disclosure.",
    "The California Consumer Privacy Act (CCPA) gives consumers the right to know what personal data is collected and to opt out of its sale.",
    "Arbitration clauses in contracts require disputes to be resolved through private arbitration rather than public court litigation.",
    "The Sarbanes-Oxley Act requires public companies to maintain internal controls over financial reporting and hold executives personally accountable.",
    "Trade secret protection under the Defend Trade Secrets Act (DTSA) provides federal civil remedies for trade secret misappropriation.",
    "Non-compete agreements restrict employees from working for competitors for a specified period after leaving, though enforceability varies by jurisdiction.",
    "The Digital Millennium Copyright Act (DMCA) criminalizes circumvention of digital rights management (DRM) technologies protecting copyrighted works.",
    "Class action lawsuits allow a large group of plaintiffs with similar claims to sue a defendant collectively, improving judicial efficiency.",
    "The Foreign Corrupt Practices Act (FCPA) prohibits US companies from bribing foreign officials to obtain or retain business advantages.",
    "Force majeure clauses excuse contractual performance when extraordinary events like natural disasters, pandemics, or wars prevent fulfillment.",
    "Attorney-client privilege protects confidential communications between a lawyer and client from disclosure, encouraging open and honest legal consultation.",
    "Statute of limitations sets the maximum time after an event within which legal proceedings may be initiated, varying by offense type.",
    "The Sherman Antitrust Act prohibits monopolistic business practices and conspiracies to restrain trade, enforced by the DOJ and FTC.",
    "Intellectual property licensing agreements define the terms under which a rights holder grants another party permission to use patents, trademarks, or copyrights.",
    "Data breach notification laws require organizations to inform affected individuals and regulators within specified timeframes when personal data is compromised.",
]

# Combine all documents
corpus = tech_docs + medical_docs + finance_docs + legal_docs
domain_labels = (
    ['tech'] * len(tech_docs) +
    ['medical'] * len(medical_docs) +
    ['finance'] * len(finance_docs) +
    ['legal'] * len(legal_docs)
)

print(f"✅ Corpus created: {len(corpus)} documents")
print(f"   - Technology: {len(tech_docs)} docs")
print(f"   - Medicine:   {len(medical_docs)} docs")
print(f"   - Finance:    {len(finance_docs)} docs")
print(f"   - Legal:      {len(legal_docs)} docs")
print(f"\nAvg document length: {np.mean([len(d) for d in corpus]):.0f} characters")
print(f"Min: {min(len(d) for d in corpus)} | Max: {max(len(d) for d in corpus)} characters")

In [ ]:
#@title 🎧 Listen: Dense Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_16_dense_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Dense Retrieval with Sentence-Transformers + FAISS

Dense retrieval embeds queries and documents into the same vector space, then finds the closest document vectors to the query vector. We use FAISS (Facebook AI Similarity Search) for fast nearest-neighbor lookup.

In [ ]:
#@title 🎧 Listen: Dense Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_17_dense_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# Load the embedding model
print("Loading embedding model (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = embed_model.get_sentence_embedding_dimension()
print(f"Model loaded! Embedding dimension: {embedding_dim}")

# Embed the entire corpus
print(f"\nEmbedding {len(corpus)} documents...")
t0 = time.time()
doc_embeddings = embed_model.encode(corpus, show_progress_bar=True, normalize_embeddings=True)
doc_embeddings = np.array(doc_embeddings, dtype='float32')
embed_time = time.time() - t0
print(f"Done in {embed_time:.2f}s | Shape: {doc_embeddings.shape}")

# Build FAISS index for fast search
print("\nBuilding FAISS index...")
faiss_index = faiss.IndexFlatIP(embedding_dim)  # Inner product = cosine sim for normalized vectors
faiss_index.add(doc_embeddings)
print(f"FAISS index ready with {faiss_index.ntotal} vectors")

In [ ]:
#@title 🎧 Listen: Dense Function Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_18_dense_function_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def dense_retrieve(query: str, top_k: int = 10) -> List[Tuple[int, float]]:
    """
    Retrieve top-K documents using dense (embedding) search via FAISS.

    Returns: List of (doc_index, cosine_similarity_score) tuples.
    """
    # Encode query
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype('float32')

    # Search FAISS index
    scores, indices = faiss_index.search(query_emb, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx >= 0:  # FAISS returns -1 for empty slots
            results.append((int(idx), float(score)))

    return results


# Test dense retrieval
test_query = "How do containers work in cloud computing?"
dense_results = dense_retrieve(test_query, top_k=5)

print(f"🔍 Dense retrieval for: \"{test_query}\"\n")
for rank, (idx, score) in enumerate(dense_results, 1):
    domain = domain_labels[idx]
    doc_preview = corpus[idx][:90] + "..." if len(corpus[idx]) > 90 else corpus[idx]
    print(f"  {rank}. [{score:.4f}] [{domain}] {doc_preview}")

In [ ]:
#@title 🎧 Listen: Sparse Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_19_sparse_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 Sparse Retrieval with BM25

BM25 is the backbone of traditional search engines. It scores documents based on term frequency and inverse document frequency — no neural networks, no embeddings, just statistics.

In [ ]:
#@title 🎧 Listen: Sparse Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_20_sparse_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize the corpus for BM25 (simple whitespace + lowercase tokenization)
def tokenize(text: str) -> List[str]:
    """Simple tokenizer: lowercase, split on non-alphanumeric, remove short tokens."""
    tokens = re.findall(r'[a-z0-9][\w\-]*', text.lower())
    return [t for t in tokens if len(t) > 1]

tokenized_corpus = [tokenize(doc) for doc in corpus]

# Build BM25 index
print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print(f"BM25 index ready!")
print(f"  Vocabulary size: {len(bm25.idf)} terms")
print(f"  Average document length: {bm25.avgdl:.1f} tokens")


def sparse_retrieve(query: str, top_k: int = 10) -> List[Tuple[int, float]]:
    """
    Retrieve top-K documents using BM25 sparse search.

    Returns: List of (doc_index, bm25_score) tuples, sorted by score.
    """
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    # Get top-K indices sorted by score (descending)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:  # Only include documents with non-zero score
            results.append((int(idx), float(scores[idx])))

    return results


# Test sparse retrieval with the SAME query
sparse_results = sparse_retrieve(test_query, top_k=5)

print(f"🔍 BM25 retrieval for: \"{test_query}\"\n")
for rank, (idx, score) in enumerate(sparse_results, 1):
    domain = domain_labels[idx]
    doc_preview = corpus[idx][:90] + "..." if len(corpus[idx]) > 90 else corpus[idx]
    print(f"  {rank}. [{score:.4f}] [{domain}] {doc_preview}")

In [ ]:
#@title 🎧 Listen: Viz Dense Sparse Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_21_viz_dense_sparse_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 📊 Visualization: Dense vs Sparse on Three Query Types

Let us see exactly where each approach wins and fails. We test three query types: semantic, keyword-exact, and mixed.

In [ ]:
#@title 🎧 Listen: Viz Dense Sparse Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_22_viz_dense_sparse_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
test_queries_comparison = [
    ("automobile regulations in the US", "Semantic: no keyword overlap"),
    ("error code ERR-4021 connection timeout", "Keyword: exact identifier"),
    ("how to fix ERR-4021 in microservices", "Mixed: keyword + concept"),
]

fig, axes = plt.subplots(len(test_queries_comparison), 2, figsize=(18, 4 * len(test_queries_comparison)))

for row, (query, query_type) in enumerate(test_queries_comparison):
    dense_res = dense_retrieve(query, top_k=5)
    sparse_res = sparse_retrieve(query, top_k=5)

    for col, (results, method, color) in enumerate([
        (dense_res, "Dense (Embedding)", "#2196F3"),
        (sparse_res, "Sparse (BM25)", "#FF9800"),
    ]):
        ax = axes[row][col]
        if not results:
            ax.text(0.5, 0.5, "No results", ha='center', va='center',
                    fontsize=14, transform=ax.transAxes)
            ax.set_title(f"{method}", fontsize=11, fontweight='bold')
            continue

        labels = []
        scores = []
        bar_colors = []
        for idx, score in results:
            label = corpus[idx][:55] + "..."
            labels.append(label)
            scores.append(score)
            bar_colors.append(color)

        y_pos = range(len(labels))
        bars = ax.barh(y_pos, scores, color=bar_colors, alpha=0.75,
                       edgecolor='white', height=0.6)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel('Score')

        title = f"{method}"
        if row == 0:
            ax.set_title(title, fontsize=12, fontweight='bold')

        for bar, score in zip(bars, scores):
            ax.text(bar.get_width() + max(scores) * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f'{score:.3f}', va='center', fontsize=8)

    # Add query type label on the left
    axes[row][0].set_ylabel(query_type, fontsize=10, fontweight='bold',
                            rotation=0, labelpad=120, va='center')

plt.suptitle('Dense vs Sparse Retrieval: Where Each Method Wins',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Key observations:")
print("   - Dense wins on semantic queries (automobile → motor vehicle)")
print("   - Sparse wins on exact-match queries (ERR-4021)")
print("   - Neither alone handles mixed queries perfectly")

In [ ]:
#@title 🎧 Listen: Hybrid Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_23_hybrid_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 Hybrid Search with Reciprocal Rank Fusion

Now we combine dense and sparse results. The idea is simple: if a document ranks high in *both* systems, it is probably very relevant.

In [ ]:
#@title 🎧 Listen: Rrf Hybrid Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_24_rrf_hybrid_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[int, float]]],
    k: int = 60,
    top_k: int = 10,
) -> List[Tuple[int, float]]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: List of ranked result lists, each containing (doc_index, score) tuples
        k: Smoothing constant (default 60, as per original RRF paper)
        top_k: Number of results to return

    Returns:
        Merged ranked list of (doc_index, rrf_score) tuples
    """
    rrf_scores = {}

    for ranked_list in ranked_lists:
        for rank, (doc_idx, _) in enumerate(ranked_list, start=1):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
            rrf_scores[doc_idx] += 1.0 / (k + rank)

    # Sort by RRF score (descending) and return top-K
    sorted_results = sorted(rrf_scores.items(), key=lambda x: -x[1])[:top_k]
    return sorted_results


def hybrid_retrieve(query: str, top_k: int = 10, rrf_k: int = 60) -> List[Tuple[int, float]]:
    """
    Hybrid retrieval: combine dense and sparse results with RRF.
    """
    # Get candidates from both systems (retrieve more than top_k for better fusion)
    dense_results = dense_retrieve(query, top_k=50)
    sparse_results = sparse_retrieve(query, top_k=50)

    # Fuse with RRF
    hybrid_results = reciprocal_rank_fusion(
        [dense_results, sparse_results],
        k=rrf_k,
        top_k=top_k,
    )

    return hybrid_results


# Test hybrid retrieval
for query, query_type in test_queries_comparison:
    hybrid_results = hybrid_retrieve(query, top_k=3)
    print(f"\n🔍 Hybrid [{query_type}]: \"{query}\"")
    for rank, (idx, score) in enumerate(hybrid_results, 1):
        domain = domain_labels[idx]
        doc_preview = corpus[idx][:80] + "..."
        print(f"   {rank}. [RRF={score:.5f}] [{domain}] {doc_preview}")

In [ ]:
#@title 🎧 Listen: Reranking Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_25_reranking_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.5 Cross-Encoder Reranking

The final stage: take our hybrid top-50 candidates and re-score them with a cross-encoder. We use `cross-encoder/ms-marco-MiniLM-L-6-v2`, a model specifically trained on the MS MARCO passage ranking dataset.

In [ ]:
#@title 🎧 Listen: Reranking Pipeline Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_26_reranking_pipeline_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading cross-encoder (ms-marco-MiniLM-L-6-v2)...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder loaded!")


def rerank_with_cross_encoder(
    query: str,
    candidates: List[Tuple[int, float]],
    top_k: int = 5,
) -> List[Tuple[int, float]]:
    """
    Rerank candidates using a cross-encoder model.

    Takes (doc_index, previous_score) tuples, scores each (query, doc) pair
    with the cross-encoder, and returns reranked results.
    """
    if not candidates:
        return []

    # Create (query, document) pairs
    pairs = [(query, corpus[idx]) for idx, _ in candidates]

    # Score with cross-encoder
    ce_scores = cross_encoder.predict(pairs)

    # Combine with original indices
    reranked = [(idx, float(score)) for (idx, _), score in zip(candidates, ce_scores)]

    # Sort by cross-encoder score (descending)
    reranked.sort(key=lambda x: -x[1])

    return reranked[:top_k]


def full_pipeline_retrieve(query: str, top_k: int = 5) -> Dict:
    """
    Complete two-stage retrieval pipeline:
      Stage 1: Hybrid (dense + sparse + RRF) → top 50
      Stage 2: Cross-encoder reranking → top K

    Returns dict with results from each stage for comparison.
    """
    t0 = time.time()

    # Stage 1a: Dense retrieval
    dense_results = dense_retrieve(query, top_k=50)
    t_dense = time.time() - t0

    # Stage 1b: Sparse retrieval
    t1 = time.time()
    sparse_results = sparse_retrieve(query, top_k=50)
    t_sparse = time.time() - t1

    # Stage 1c: Hybrid fusion
    t2 = time.time()
    hybrid_results = reciprocal_rank_fusion(
        [dense_results, sparse_results], k=60, top_k=50
    )
    t_hybrid = time.time() - t2

    # Stage 2: Cross-encoder reranking
    t3 = time.time()
    reranked_results = rerank_with_cross_encoder(query, hybrid_results, top_k=top_k)
    t_rerank = time.time() - t3

    return {
        'query': query,
        'dense': dense_results[:top_k],
        'sparse': sparse_results[:top_k],
        'hybrid': hybrid_results[:top_k],
        'reranked': reranked_results,
        'timings': {
            'dense': t_dense,
            'sparse': t_sparse,
            'hybrid': t_hybrid,
            'rerank': t_rerank,
            'total': t_dense + t_sparse + t_hybrid + t_rerank,
        },
    }


# Test the full pipeline
test_result = full_pipeline_retrieve("How do containers work in cloud computing?", top_k=5)
print(f"\n🔍 Full pipeline for: \"{test_result['query']}\"")
print(f"\nTimings: dense={test_result['timings']['dense']*1000:.1f}ms, "
      f"sparse={test_result['timings']['sparse']*1000:.1f}ms, "
      f"hybrid={test_result['timings']['hybrid']*1000:.1f}ms, "
      f"rerank={test_result['timings']['rerank']*1000:.1f}ms, "
      f"total={test_result['timings']['total']*1000:.1f}ms")

print(f"\nTop-5 reranked results:")
for rank, (idx, score) in enumerate(test_result['reranked'], 1):
    doc_preview = corpus[idx][:80] + "..."
    print(f"  {rank}. [{score:.4f}] [{domain_labels[idx]}] {doc_preview}")

In [ ]:
#@title 🎧 Listen: Viz All Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_27_viz_all_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 📊 Visualization: All Four Methods Side-by-Side

In [ ]:
#@title 🎧 Listen: Viz All Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_28_viz_all_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Compare all 4 methods on the three test query types
fig, axes = plt.subplots(3, 4, figsize=(22, 12))

method_colors = {
    'dense': '#2196F3',
    'sparse': '#FF9800',
    'hybrid': '#4CAF50',
    'reranked': '#E91E63',
}
method_names = {
    'dense': 'Dense\n(Embedding)',
    'sparse': 'Sparse\n(BM25)',
    'hybrid': 'Hybrid\n(RRF)',
    'reranked': 'Reranked\n(Cross-Encoder)',
}

for row, (query, query_type) in enumerate(test_queries_comparison):
    result = full_pipeline_retrieve(query, top_k=5)

    for col, method in enumerate(['dense', 'sparse', 'hybrid', 'reranked']):
        ax = axes[row][col]
        results = result[method]

        if not results:
            ax.text(0.5, 0.5, "No results", ha='center', va='center',
                    fontsize=12, transform=ax.transAxes)
        else:
            labels = [corpus[idx][:45] + "..." for idx, _ in results[:5]]
            scores = [score for _, score in results[:5]]
            color = method_colors[method]

            y_pos = range(len(labels))
            bars = ax.barh(y_pos, scores, color=color, alpha=0.75,
                           edgecolor='white', height=0.6)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(labels, fontsize=6)
            ax.invert_yaxis()

            for bar, score in zip(bars, scores):
                ax.text(bar.get_width() + max(scores) * 0.02,
                        bar.get_y() + bar.get_height() / 2,
                        f'{score:.3f}', va='center', fontsize=7)

        if row == 0:
            ax.set_title(method_names[method], fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(query_type, fontsize=9, fontweight='bold',
                          rotation=0, labelpad=110, va='center')

plt.suptitle('Four Retrieval Strategies: Head-to-Head Comparison',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_29_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. ✍️ Your Turn (TODO Sections)

### 🔧 TODO 1: Implement Reciprocal Rank Fusion from Scratch

We used a pre-built RRF function above. Now implement it yourself — and then explore how the $k$ parameter changes the ranking.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_30_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def rrf_from_scratch(
    dense_ranked: List[Tuple[int, float]],
    sparse_ranked: List[Tuple[int, float]],
    k: int = 60,
    top_k: int = 10,
) -> List[Tuple[int, float]]:
    """
    Implement Reciprocal Rank Fusion from scratch.

    Formula: RRF(d) = 1/(k + rank_dense(d)) + 1/(k + rank_sparse(d))

    For documents appearing in only ONE list, use only that list's contribution.

    Args:
        dense_ranked:  List of (doc_index, dense_score) sorted by dense_score descending
        sparse_ranked: List of (doc_index, bm25_score) sorted by bm25_score descending
        k: Smoothing constant (try values: 1, 10, 60, 200)
        top_k: Number of results to return

    Returns:
        List of (doc_index, rrf_score) tuples, sorted by rrf_score descending

    Hints:
        - Create a dict to accumulate RRF scores per document
        - enumerate() gives you the rank (starting from 1, not 0!)
        - Documents may appear in one list but not the other — handle this correctly
        - Use sorted() with a key function to sort the final results
    """
    # ============ TODO ============
    # Step 1: Create a dict mapping doc_idx → rrf_score
    # Step 2: Iterate through dense_ranked with enumerate (rank starts at 1)
    #         For each doc, add 1/(k + rank) to its score
    # Step 3: Iterate through sparse_ranked with enumerate (rank starts at 1)
    #         For each doc, add 1/(k + rank) to its score
    # Step 4: Sort by score descending, return top_k
    # ==============================

    rrf_scores = {}  # YOUR CODE HERE

    # Process dense results
    # YOUR CODE HERE

    # Process sparse results
    # YOUR CODE HERE

    # Sort and return top-k
    sorted_results = ???  # YOUR CODE HERE

    return sorted_results

In [ ]:
#@title 🎧 Listen: Todo1 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_31_todo1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ✅ Verification for TODO 1

# Test case: known inputs with predictable output
test_dense = [(0, 0.95), (1, 0.90), (2, 0.85)]
test_sparse = [(1, 12.5), (2, 10.0), (3, 8.0)]

result_60 = rrf_from_scratch(test_dense, test_sparse, k=60, top_k=4)

# Doc 1 should be top: appears in BOTH lists
# Doc 1: rank 2 in dense + rank 1 in sparse
# Doc 1: 1/(60+2) + 1/(60+1) = 1/62 + 1/61
# Doc 0: rank 1 in dense only → 1/(60+1) = 1/61

result_dict = dict(result_60)

# Doc 1 should have highest score (appears in both lists at good ranks)
assert result_60[0][0] == 1, f"Expected doc 1 at top, got doc {result_60[0][0]}"

# Doc 1 score should be approximately 1/62 + 1/61
expected_doc1 = 1/62 + 1/61
assert abs(result_dict[1] - expected_doc1) < 1e-6, \
    f"Doc 1 score: expected {expected_doc1:.6f}, got {result_dict[1]:.6f}"

# Doc 0 should only have dense contribution (rank 1)
expected_doc0 = 1/61
assert abs(result_dict[0] - expected_doc0) < 1e-6, \
    f"Doc 0 score: expected {expected_doc0:.6f}, got {result_dict[0]:.6f}"

# Should return 4 results (union of all unique docs)
assert len(result_60) == 4, f"Expected 4 results, got {len(result_60)}"

print("✅ All verification checks passed!")
print(f"\nResults (k=60):")
for idx, score in result_60:
    in_dense = "✓" if idx in [d[0] for d in test_dense] else "✗"
    in_sparse = "✓" if idx in [d[0] for d in test_sparse] else "✗"
    print(f"  Doc {idx}: RRF={score:.6f}  (dense:{in_dense}, sparse:{in_sparse})")

In [ ]:
#@title 🎧 Listen: Viz K Rrf Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_32_viz_k_rrf_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 📊 Visualization: How k Affects RRF Rankings

In [ ]:
#@title 🎧 Listen: Viz K Rrf Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_33_viz_k_rrf_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Now let us see how k changes the ranking behavior
k_values = [1, 5, 10, 30, 60, 100, 200, 500]

# Use a real query with both dense and sparse results
demo_query = "how to fix ERR-4021 in microservices"
demo_dense = dense_retrieve(demo_query, top_k=20)
demo_sparse = sparse_retrieve(demo_query, top_k=20)

# Track RRF scores for top documents across different k values
ref_results = reciprocal_rank_fusion([demo_dense, demo_sparse], k=60, top_k=5)
tracked_docs = [idx for idx, _ in ref_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: RRF scores vs k for tracked documents
for doc_idx in tracked_docs:
    scores_by_k = []
    for kv in k_values:
        rrf_result = reciprocal_rank_fusion([demo_dense, demo_sparse], k=kv, top_k=50)
        rrf_dict = dict(rrf_result)
        scores_by_k.append(rrf_dict.get(doc_idx, 0.0))

    label = corpus[doc_idx][:40] + "..."
    ax1.plot(k_values, scores_by_k, 'o-', label=label, linewidth=2, markersize=5)

ax1.set_xlabel('k parameter', fontsize=12)
ax1.set_ylabel('RRF Score', fontsize=12)
ax1.set_title('RRF Score vs k Parameter\n(Higher k = more democratic ranking)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=7, loc='upper right')
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

# Panel 2: Rank stability across k values
rank_data = {doc_idx: [] for doc_idx in tracked_docs}
for kv in k_values:
    rrf_result = reciprocal_rank_fusion([demo_dense, demo_sparse], k=kv, top_k=50)
    rank_dict = {idx: rank for rank, (idx, _) in enumerate(rrf_result, 1)}
    for doc_idx in tracked_docs:
        rank_data[doc_idx].append(rank_dict.get(doc_idx, 50))

for doc_idx in tracked_docs:
    label = corpus[doc_idx][:40] + "..."
    ax2.plot(k_values, rank_data[doc_idx], 'o-', label=label, linewidth=2, markersize=5)

ax2.set_xlabel('k parameter', fontsize=12)
ax2.set_ylabel('Rank (lower is better)', fontsize=12)
ax2.set_title('Rank Stability Across k Values\n(Stable ranking = robust fusion)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=7, loc='upper left')
ax2.set_xscale('log')
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key insight: As k increases, the ranking becomes more stable")
print("   because rank differences matter less. k=60 is the standard choice")
print("   from the original RRF paper.")

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_34_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 🔧 TODO 2: Build a Query Classifier

Build a classifier that automatically decides which retrieval strategy to use based on query characteristics. Some queries are best served by dense search (semantic/natural language), others by sparse search (keyword/identifier), and some need hybrid.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_35_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def classify_query(query: str) -> str:
    """
    Classify a query into one of three retrieval strategies:
      - "sparse":  Query contains exact identifiers, error codes, model names,
                    version numbers, or technical terms that need exact matching
      - "dense":   Query is natural language, asks conceptual questions,
                    or uses synonyms/paraphrasing
      - "hybrid":  Query mixes both — has identifiers AND conceptual questions

    Args:
        query: The search query string

    Returns:
        One of "sparse", "dense", or "hybrid"

    Strategy:
        1. Check for patterns indicating exact-match needs:
           - Error codes (ERR-XXXX, HTTP 4xx/5xx)
           - Version numbers (v2.1, Python 3.10)
           - Quoted strings
           - ALL-CAPS tokens of 3+ letters (acronyms like GDPR, HIPAA, VaR)
           - Hyphenated technical identifiers
        2. Check for patterns indicating semantic needs:
           - Question words (how, what, why, explain)
           - Natural language phrasing (longer queries, conversational)
        3. If BOTH patterns present → "hybrid"
           If only exact-match patterns → "sparse"
           If only semantic patterns → "dense"
           Default: "hybrid" (safest fallback)

    Hints:
        - Use regex patterns like r'ERR-\d+', r'[A-Z]{3,}', r'v\d+[\.\d]*'
        - len(query.split()) > 5 suggests natural language
        - Question words: how, what, why, when, explain, describe, compare
    """
    # ============ TODO ============
    # Step 1: Define regex patterns for exact-match indicators
    #         error_codes = re.findall(r'ERR-\d+|HTTP\s*[45]\d{2}', query)
    #         acronyms = re.findall(r'\b[A-Z]{3,}\b', query)
    #         versions = re.findall(r'v\d+[\.\d]*|\d+\.\d+', query)
    #         quoted = re.findall(r'"[^"]*"', query)
    # Step 2: Define indicators for semantic/conceptual queries
    #         question_words = {'how', 'what', 'why', 'when', 'explain', 'describe', 'compare'}
    #         has_question = any(w in query.lower().split() for w in question_words)
    #         is_natural_language = len(query.split()) > 5
    # Step 3: Combine signals
    #         has_exact = bool(error_codes or acronyms or versions or quoted)
    #         has_semantic = has_question or is_natural_language
    #         if has_exact and has_semantic: return "hybrid"
    #         if has_exact: return "sparse"
    #         if has_semantic: return "dense"
    #         return "hybrid"
    # ==============================

    strategy = ???  # YOUR CODE HERE
    return strategy

In [ ]:
#@title 🎧 Listen: Todo2 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_36_todo2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ✅ Verification for TODO 2

test_queries_classify = [
    # Should be SPARSE (exact identifiers)
    ("ERR-4021 timeout", "sparse"),
    ("ERR-5033 TLS handshake", "sparse"),
    ("HIPAA compliance", "sparse"),
    ("GDPR data rights", "sparse"),
    ("OAuth 2.0 token", "sparse"),
    ("Basel III requirements", "sparse"),

    # Should be DENSE (natural language / conceptual)
    ("How do containers help with deployment?", "dense"),
    ("What is the best way to prevent overfitting in neural networks?", "dense"),
    ("Explain the difference between microservices and monoliths", "dense"),
    ("Why do distributed systems need consensus protocols?", "dense"),
    ("How can machine learning improve medical diagnosis?", "dense"),

    # Should be HYBRID (mixed)
    ("How to fix ERR-4021 in a microservices architecture?", "hybrid"),
    ("Explain what GDPR means for data privacy in healthcare", "hybrid"),
    ("What are the HIPAA requirements for telemedicine platforms?", "hybrid"),
    ("How does the Black-Scholes model handle volatility?", "hybrid"),
]

correct = 0
results_table = []

for query, expected in test_queries_classify:
    predicted = classify_query(query)
    is_correct = predicted == expected
    correct += int(is_correct)
    mark = "✅" if is_correct else "❌"
    results_table.append((mark, query[:50], expected, predicted))

print(f"Query Classification Results: {correct}/{len(test_queries_classify)} correct")
print(f"{'':>3} {'Query':<52} {'Expected':<10} {'Predicted':<10}")
print("-" * 78)
for mark, query, expected, predicted in results_table:
    print(f" {mark} {query:<52} {expected:<10} {predicted:<10}")

accuracy = correct / len(test_queries_classify) * 100
if accuracy >= 80:
    print(f"\n✅ Great! {accuracy:.0f}% accuracy — your classifier works well!")
elif accuracy >= 60:
    print(f"\n⚠️ {accuracy:.0f}% accuracy — decent, but check the misclassified queries")
else:
    print(f"\n❌ {accuracy:.0f}% accuracy — review your classification logic")

In [ ]:
#@title 🎧 Listen: Section6 Transition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_37_section6_transition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. 🔗 Putting It All Together

Now let us build the evaluation harness. We create a labeled test set of 20 query-document pairs and measure Recall@5, MRR (Mean Reciprocal Rank), and nDCG (normalized Discounted Cumulative Gain) for each retrieval stage.

In [ ]:
#@title 🎧 Listen: Labeled Test Metrics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_38_labeled_test_metrics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ========================================
# LABELED TEST SET: 20 queries with ground-truth relevant documents
# ========================================

labeled_queries = [
    # (query, set of relevant document indices)
    ("container orchestration for cloud applications", {0, 1}),           # Kubernetes, Docker
    ("real-time communication protocols for web apps", {4, 5}),           # WebSocket, gRPC
    ("NoSQL document database for flexible schemas", {8}),                # MongoDB
    ("error code ERR-4021 connection timeout fix", {14}),                 # ERR-4021
    ("error code ERR-5033 TLS certificate mismatch", {15}),              # ERR-5033
    ("infrastructure as code across cloud providers", {10}),              # Terraform
    ("how microservices communicate with each other", {12, 13}),         # Microservices, service mesh
    ("authentication tokens for stateless APIs", {17, 18}),              # OAuth, JWT
    ("first-line treatment for type 2 diabetes", {30}),                  # Metformin
    ("gene editing technology using guide RNA", {31}),                    # CRISPR
    ("vaccines that use messenger RNA technology", {32}),                 # mRNA vaccines
    ("immunotherapy that blocks PD-1 checkpoint", {33}),                 # PD-1 inhibitors
    ("how to measure risk-adjusted portfolio returns", {52}),            # Sharpe ratio
    ("predicting recessions from interest rate data", {56}),             # Yield curve
    ("privacy law that gives EU citizens data rights", {60}),            # GDPR
    ("US law protecting internet platforms from liability", {61}),       # Section 230
    ("AI regulation framework with risk-based classification", {62}),    # EU AI Act
    ("protecting patient health information privacy", {65}),             # HIPAA
    ("distributed system consistency and availability tradeoff", {26}),  # CAP theorem
    ("automated build test deploy pipeline", {11}),                      # CI/CD
]

def compute_recall_at_k(retrieved_indices: List[int], relevant_indices: set, k: int = 5) -> float:
    """Recall@K: fraction of relevant docs found in top-K."""
    top_k_set = set(retrieved_indices[:k])
    hits = len(top_k_set & relevant_indices)
    return hits / len(relevant_indices) if relevant_indices else 0.0

def compute_mrr(retrieved_indices: List[int], relevant_indices: set) -> float:
    """Mean Reciprocal Rank: 1/rank of first relevant document."""
    for rank, idx in enumerate(retrieved_indices, 1):
        if idx in relevant_indices:
            return 1.0 / rank
    return 0.0

def compute_ndcg(retrieved_indices: List[int], relevant_indices: set, k: int = 5) -> float:
    """
    Normalized Discounted Cumulative Gain @ K.
    Binary relevance: 1 if relevant, 0 otherwise.
    """
    dcg = 0.0
    for i, idx in enumerate(retrieved_indices[:k]):
        rel = 1.0 if idx in relevant_indices else 0.0
        dcg += rel / np.log2(i + 2)  # i+2 because rank starts at 1, log2(1)=0

    # Ideal DCG: all relevant docs at top
    ideal_rels = sorted([1.0] * min(len(relevant_indices), k) +
                        [0.0] * max(0, k - len(relevant_indices)), reverse=True)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))

    return dcg / idcg if idcg > 0 else 0.0


print(f"✅ Evaluation harness ready with {len(labeled_queries)} labeled queries")
print(f"   Metrics: Recall@5, MRR, nDCG@5")

In [ ]:
#@title 🎧 Listen: Evaluate All Methods
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_39_evaluate_all_methods.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ========================================
# EVALUATE ALL 4 METHODS
# ========================================

methods = {
    'Dense': lambda q: [idx for idx, _ in dense_retrieve(q, top_k=50)],
    'Sparse': lambda q: [idx for idx, _ in sparse_retrieve(q, top_k=50)],
    'Hybrid': lambda q: [idx for idx, _ in hybrid_retrieve(q, top_k=50)],
    'Reranked': lambda q: [idx for idx, _ in rerank_with_cross_encoder(
        q, hybrid_retrieve(q, top_k=50), top_k=50
    )],
}

# Store per-query metrics
all_metrics = {method: {'recall@5': [], 'mrr': [], 'ndcg@5': []}
               for method in methods}

print("Evaluating all 4 methods on 20 labeled queries...")
print("=" * 70)

for method_name, retrieve_fn in methods.items():
    for query, relevant_indices in labeled_queries:
        retrieved = retrieve_fn(query)

        recall = compute_recall_at_k(retrieved, relevant_indices, k=5)
        mrr = compute_mrr(retrieved, relevant_indices)
        ndcg = compute_ndcg(retrieved, relevant_indices, k=5)

        all_metrics[method_name]['recall@5'].append(recall)
        all_metrics[method_name]['mrr'].append(mrr)
        all_metrics[method_name]['ndcg@5'].append(ndcg)

# Print summary table
print(f"\n{'Method':<12} {'Recall@5':>10} {'MRR':>10} {'nDCG@5':>10}")
print("-" * 44)
for method in methods:
    r5 = np.mean(all_metrics[method]['recall@5'])
    mrr = np.mean(all_metrics[method]['mrr'])
    ndcg = np.mean(all_metrics[method]['ndcg@5'])
    print(f"{method:<12} {r5:>10.3f} {mrr:>10.3f} {ndcg:>10.3f}")

# Find the winner
best_method = max(methods.keys(), key=lambda m: np.mean(all_metrics[m]['ndcg@5']))
print(f"\n🏆 Best method by nDCG@5: {best_method}")

In [ ]:
#@title 🎧 Listen: Section7 Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_40_section7_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. 📊 Training and Results — Four-Panel Comparison

In [ ]:
#@title 🎧 Listen: Four Panel Viz Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_41_four_panel_viz_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
method_colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
method_list = list(methods.keys())

# ──────────────────────────────────────────────
# Panel 1: Recall@5 by Method (grouped by query domain)
# ──────────────────────────────────────────────
ax1 = axes[0][0]

# Group queries by domain based on relevant doc indices
domain_groups = {'Tech': [], 'Medical': [], 'Finance': [], 'Legal': []}
for i, (query, relevant) in enumerate(labeled_queries):
    min_idx = min(relevant)
    if min_idx < len(tech_docs):
        domain_groups['Tech'].append(i)
    elif min_idx < len(tech_docs) + len(medical_docs):
        domain_groups['Medical'].append(i)
    elif min_idx < len(tech_docs) + len(medical_docs) + len(finance_docs):
        domain_groups['Finance'].append(i)
    else:
        domain_groups['Legal'].append(i)

x = np.arange(len(domain_groups))
width = 0.18

for j, method in enumerate(method_list):
    recalls_by_domain = []
    for domain, query_indices in domain_groups.items():
        domain_recalls = [all_metrics[method]['recall@5'][i] for i in query_indices]
        recalls_by_domain.append(np.mean(domain_recalls) if domain_recalls else 0)

    bars = ax1.bar(x + j * width, recalls_by_domain, width,
                   label=method, color=method_colors[j], alpha=0.8, edgecolor='white')

ax1.set_xticks(x + 1.5 * width)
ax1.set_xticklabels(list(domain_groups.keys()), fontsize=11)
ax1.set_ylabel('Recall@5', fontsize=12)
ax1.set_title('Panel 1: Recall@5 by Domain and Method', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.grid(axis='y', alpha=0.3)

# ──────────────────────────────────────────────
# Panel 2: MRR Comparison (overall)
# ──────────────────────────────────────────────
ax2 = axes[0][1]

mrr_means = [np.mean(all_metrics[m]['mrr']) for m in method_list]
mrr_stds = [np.std(all_metrics[m]['mrr']) for m in method_list]

bars2 = ax2.bar(method_list, mrr_means, yerr=mrr_stds, capsize=5,
                color=method_colors, alpha=0.8, edgecolor='white')
ax2.set_ylabel('Mean Reciprocal Rank (MRR)', fontsize=12)
ax2.set_title('Panel 2: MRR Comparison (Higher = Better)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.15)
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars2, mrr_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)

# ──────────────────────────────────────────────
# Panel 3: Latency Comparison
# ──────────────────────────────────────────────
ax3 = axes[1][0]

# Measure latency for each method
latency_results = {method: [] for method in method_list}

for query, _ in labeled_queries[:5]:  # Use first 5 queries for timing
    # Dense
    t0 = time.time()
    dense_retrieve(query, top_k=50)
    latency_results['Dense'].append((time.time() - t0) * 1000)

    # Sparse
    t0 = time.time()
    sparse_retrieve(query, top_k=50)
    latency_results['Sparse'].append((time.time() - t0) * 1000)

    # Hybrid
    t0 = time.time()
    hybrid_retrieve(query, top_k=50)
    latency_results['Hybrid'].append((time.time() - t0) * 1000)

    # Reranked
    t0 = time.time()
    candidates = hybrid_retrieve(query, top_k=50)
    rerank_with_cross_encoder(query, candidates, top_k=5)
    latency_results['Reranked'].append((time.time() - t0) * 1000)

latency_means = [np.mean(latency_results[m]) for m in method_list]
latency_stds = [np.std(latency_results[m]) for m in method_list]

bars3 = ax3.bar(method_list, latency_means, yerr=latency_stds, capsize=5,
                color=method_colors, alpha=0.8, edgecolor='white')
ax3.set_ylabel('Latency (ms)', fontsize=12)
ax3.set_title('Panel 3: Latency Comparison (Lower = Better)', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val in zip(bars3, latency_means):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(latency_means) * 0.02,
             f'{val:.0f}ms', ha='center', fontweight='bold', fontsize=11)

# ──────────────────────────────────────────────
# Panel 4: Score Distribution — Relevant vs Irrelevant
# ──────────────────────────────────────────────
ax4 = axes[1][1]

# Collect cross-encoder scores for relevant vs irrelevant docs
relevant_scores = []
irrelevant_scores = []

for query, relevant_indices in labeled_queries[:10]:
    candidates = hybrid_retrieve(query, top_k=20)
    if not candidates:
        continue
    pairs = [(query, corpus[idx]) for idx, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)

    for (idx, _), score in zip(candidates, ce_scores):
        if idx in relevant_indices:
            relevant_scores.append(score)
        else:
            irrelevant_scores.append(score)

bins = np.linspace(
    min(min(relevant_scores, default=0), min(irrelevant_scores, default=0)),
    max(max(relevant_scores, default=1), max(irrelevant_scores, default=1)),
    30
)

ax4.hist(irrelevant_scores, bins=bins, alpha=0.6, color='#EF5350', label='Irrelevant docs',
         edgecolor='white', density=True)
ax4.hist(relevant_scores, bins=bins, alpha=0.7, color='#66BB6A', label='Relevant docs',
         edgecolor='white', density=True)
ax4.set_xlabel('Cross-Encoder Score', fontsize=12)
ax4.set_ylabel('Density', fontsize=12)
ax4.set_title('Panel 4: Score Distribution\n(Relevant vs Irrelevant)', fontsize=13, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(axis='y', alpha=0.3)

# Add separation annotation
if relevant_scores and irrelevant_scores:
    rel_mean = np.mean(relevant_scores)
    irr_mean = np.mean(irrelevant_scores)
    ax4.axvline(rel_mean, color='#2E7D32', linestyle='--', linewidth=2, alpha=0.8)
    ax4.axvline(irr_mean, color='#C62828', linestyle='--', linewidth=2, alpha=0.8)
    ax4.annotate(f'Separation = {rel_mean - irr_mean:.2f}',
                 xy=((rel_mean + irr_mean)/2, ax4.get_ylim()[1] * 0.9),
                 fontsize=11, fontweight='bold', ha='center',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray'))

plt.suptitle('Retrieval Strategy Evaluation: Complete Comparison',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Summary:")
print(f"   - Reranked hybrid achieves the best quality (nDCG, MRR, Recall)")
print(f"   - BM25 is the fastest by far (pure computation, no neural network)")
print(f"   - The cross-encoder clearly separates relevant from irrelevant (Panel 4)")
print(f"   - The quality/latency tradeoff is the core engineering decision")

In [ ]:
#@title 🎧 Listen: Interactive Tool Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_42_interactive_tool_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. 🖥️ Final Output — Interactive Retrieval Comparison Tool

Enter any query and see side-by-side results from all four methods with scores and matching term highlights.

In [ ]:
#@title 🎧 Listen: Interactive Tool Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_43_interactive_tool_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def highlight_matching_terms(text: str, query: str, max_len: int = 120) -> str:
    """Add ** around query terms found in the text for emphasis."""
    query_terms = set(re.findall(r'[a-z0-9][\w\-]*', query.lower()))
    # Remove common stopwords
    stopwords = {'the', 'a', 'an', 'is', 'are', 'how', 'what', 'why', 'do', 'does',
                 'to', 'for', 'in', 'on', 'of', 'and', 'or', 'with', 'from', 'by'}
    query_terms -= stopwords

    words = text.split()
    highlighted = []
    for word in words:
        word_clean = re.sub(r'[^\w\-]', '', word.lower())
        if word_clean in query_terms:
            highlighted.append(f"**{word}**")
        else:
            highlighted.append(word)

    result = ' '.join(highlighted)
    if len(result) > max_len:
        result = result[:max_len] + "..."
    return result


def interactive_comparison(query: str):
    """
    Run all 4 retrieval methods on a query and display results side-by-side.
    """
    print("=" * 90)
    print(f"  QUERY: \"{query}\"")
    print("=" * 90)

    # Run all methods
    dense_res = dense_retrieve(query, top_k=5)
    sparse_res = sparse_retrieve(query, top_k=5)
    hybrid_res = hybrid_retrieve(query, top_k=5)
    reranked_res = rerank_with_cross_encoder(query, hybrid_retrieve(query, top_k=50), top_k=5)

    methods_results = [
        ("🔵 DENSE (Embedding)", dense_res),
        ("🟠 SPARSE (BM25)", sparse_res),
        ("🟢 HYBRID (RRF)", hybrid_res),
        ("🔴 RERANKED (Cross-Encoder)", reranked_res),
    ]

    for method_name, results in methods_results:
        print(f"\n  {method_name}")
        print(f"  {'─' * 80}")
        for rank, (idx, score) in enumerate(results[:5], 1):
            domain = domain_labels[idx]
            text = highlight_matching_terms(corpus[idx], query)
            print(f"  {rank}. [{score:>8.4f}] [{domain:>8}] {text}")

    print(f"\n{'=' * 90}")


# Demo with several queries
demo_queries = [
    "How do containers help with cloud deployment?",
    "error code ERR-4021 TCP timeout",
    "What privacy regulations protect health data in Europe?",
    "Monte Carlo risk simulation for investment portfolios",
    "explain how mRNA vaccines trigger immune response",
]

for query in demo_queries:
    interactive_comparison(query)
    print()

In [ ]:
#@title 🎧 Listen: Heatmap Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_44_heatmap_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 📊 Final Visualization: Method Agreement Heatmap

Which methods agree on the same documents? High agreement between dense and sparse means the document is a strong match on both axes.

In [ ]:
#@title 🎧 Listen: Heatmap Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_45_heatmap_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# For each demo query, compute method agreement
fig, axes = plt.subplots(1, len(demo_queries), figsize=(5 * len(demo_queries), 5))

for q_idx, query in enumerate(demo_queries):
    ax = axes[q_idx]

    dense_set = set(idx for idx, _ in dense_retrieve(query, top_k=10))
    sparse_set = set(idx for idx, _ in sparse_retrieve(query, top_k=10))
    hybrid_set = set(idx for idx, _ in hybrid_retrieve(query, top_k=10))
    reranked_set = set(idx for idx, _ in rerank_with_cross_encoder(
        query, hybrid_retrieve(query, top_k=50), top_k=10))

    sets = [dense_set, sparse_set, hybrid_set, reranked_set]
    method_labels = ['Dense', 'Sparse', 'Hybrid', 'Reranked']

    # Compute Jaccard similarity between each pair
    n = len(sets)
    agreement = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if len(sets[i] | sets[j]) > 0:
                agreement[i][j] = len(sets[i] & sets[j]) / len(sets[i] | sets[j])
            else:
                agreement[i][j] = 0.0

    im = ax.imshow(agreement, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_xticklabels(method_labels, fontsize=8, rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(method_labels, fontsize=8)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{agreement[i][j]:.2f}', ha='center', va='center',
                    fontsize=8, fontweight='bold',
                    color='white' if agreement[i][j] > 0.5 else 'black')

    query_short = query[:25] + "..." if len(query) > 25 else query
    ax.set_title(f'"{query_short}"', fontsize=9, fontweight='bold')

plt.suptitle('Method Agreement (Jaccard Similarity of Top-10 Results)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\n💡 High agreement between Hybrid and Reranked is expected — reranking")
print("   mostly reorders the hybrid results rather than introducing new documents.")
print("   Low Dense-Sparse agreement confirms they find DIFFERENT relevant documents.")

In [ ]:
#@title 🎧 Listen: Reflection Questions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_46_reflection_questions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. 🎯 Reflection and Next Steps

### 🤔 Reflection Questions

Take a moment to think about these before moving on:

**1. When does each strategy win?**

Consider: You are building a customer support search engine. Some queries are "How do I reset my password?" (semantic) while others are "Order #ORD-78421 status" (keyword). What would you choose as the default retrieval strategy, and why?

*Think about it...*

> **Our take:** Hybrid with cross-encoder reranking is the safest default for any system where query types are unpredictable. The cost of running both dense and sparse is minimal compared to the cost of missing relevant documents. Only optimize to a single method when you have strong evidence that your query distribution is homogeneous.

**2. The cost of reranking:**

Cross-encoder reranking on 50 candidates adds ~500ms-2s of latency. When is this acceptable, and when should you skip it?

*Think about it...*

> **Our take:** Skip reranking for (a) real-time autocomplete where latency must be <50ms, (b) when the hybrid results already have high confidence, (c) when cost per query matters more than quality (high-volume, low-stakes search). Use reranking for (a) knowledge-base search where accuracy is critical, (b) any pipeline feeding an LLM (bad retrieval = hallucination), (c) when the candidate pool is heterogeneous and initial rankings are unreliable.

**3. Tuning the hybrid alpha parameter:**

Some systems use a weighted combination instead of RRF: $\text{score}(d) = \alpha \cdot \text{dense}(d) + (1 - \alpha) \cdot \text{sparse}(d)$. How would you determine the right $\alpha$ for your application?

*Think about it...*

> **Our take:** Start with $\alpha = 0.5$ (equal weight). Then create a labeled evaluation set of ~50-100 queries representative of your traffic. Sweep $\alpha$ from 0.0 to 1.0 in increments of 0.1, measuring nDCG@5 at each value. If your queries are mostly semantic (natural language), $\alpha$ will be higher (~0.7). If keyword-heavy (identifiers, codes), $\alpha$ will be lower (~0.3). RRF is often preferred over alpha-weighting because it avoids the need to normalize scores across different scales.

In [ ]:
#@title 🎧 Listen: Challenges Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_47_challenges_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 🚀 Optional Challenges

In [ ]:
#@title 🎧 Listen: Challenges Details
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_48_challenges_details.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ──────────────────────────────────────────────
# Challenge 1: Implement Alpha-Weighted Hybrid Search
# ──────────────────────────────────────────────
#
# Instead of RRF, implement score-based fusion:
#   final_score(d) = alpha * normalize(dense_score(d)) + (1-alpha) * normalize(bm25_score(d))
#
# Normalize each score list to [0, 1] using min-max normalization.
# Then sweep alpha from 0.0 to 1.0 and plot nDCG@5 vs alpha.
# Compare to RRF: which fusion method performs better on your labeled test set?

# ──────────────────────────────────────────────
# Challenge 2: Build a Learned Sparse Retriever
# ──────────────────────────────────────────────
#
# BM25 uses raw term frequency. Can we do better with learned term weights?
# Strategy:
#   1. Use the embedding model to get token-level attention weights
#   2. Weight each term's BM25 contribution by its attention score
#   3. Compare this "attention-weighted BM25" to standard BM25
#
# This is a simplified version of what SPLADE and other learned sparse
# retrievers do in production.

print("Try these challenges! They build directly on everything you implemented above.")
print("\nChallenge 1 teaches you about score normalization — a subtle but critical issue")
print("in hybrid search (dense scores are cosine similarities in [-1,1], BM25 scores")
print("are unbounded positive values — you cannot add them directly).")
print("\nChallenge 2 is a mini-research project that bridges the gap between")
print("sparse and dense retrieval. Modern systems like SPLADE do exactly this.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_49_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### ✅ Notebook Complete!

You have built four complete retrieval strategies from scratch:

1. **Dense retrieval** — sentence-transformers + FAISS for semantic search
2. **Sparse retrieval** — BM25 for keyword matching
3. **Hybrid retrieval** — Reciprocal Rank Fusion combining both
4. **Cross-encoder reranking** — ms-marco-MiniLM-L-6-v2 for precise re-scoring

And you measured them rigorously: Recall@5, MRR, nDCG@5, latency, and score separation.

**The key takeaway:** No single retrieval strategy dominates across all query types. Dense misses exact keywords, sparse misses semantics, and only hybrid + reranking catches both. The two-stage pattern (bi-encoder top-50 → cross-encoder top-5) is the production standard because it balances quality and latency.

**Estimated time spent: ~45 minutes**

---

*Part 4 of 5 in the Vizuara series on RAG Systems (Context Engineering Course)*
*Next up: Part 5 — Building a Complete RAG Application*